# 01 — Core Guards Fundamentals

32 practical examples covering Guard init, all OnFailAction types, ValidationOutcome inspection,
history, serialization, REASK loops, and LLM integration patterns.

**Installation:**
```bash
pip install guardrails-ai openai anthropic python-dotenv
```

In [2]:
import os, json
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
import anthropic
from guardrails import Guard, AsyncGuard, OnFailAction
from guardrails.errors import ValidationError
from guardrails.hub import RegexMatch, ValidLength

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
ant = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Clients ready.')

ModuleNotFoundError: No module named 'anthropic'

## Example 01 — Minimal Guard (no validators)

In [ ]:
# Example 01: Minimal Guard creation and validate()
# Demonstrates: Guard init with no validators; ValidationOutcome fields
guard = Guard()
outcome = guard.validate('Hello, guardrails!')
print('validation_passed:', outcome.validation_passed)
print('validated_output:', outcome.validated_output)

## Example 02 — OnFailAction.EXCEPTION

In [ ]:
# Example 02: OnFailAction.EXCEPTION raises ValidationError on fail
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=100, on_fail=OnFailAction.EXCEPTION))
try:
    outcome = guard.validate('Hi')  # too short
except ValidationError as e:
    print('FAIL caught:', str(e)[:120])

outcome = guard.validate('This is a sufficiently long string that passes validation.')
print('PASS:', outcome.validated_output)

## Example 03 — OnFailAction.REASK + LLM

In [ ]:
# Example 03: OnFailAction.REASK — LLM is re-prompted to produce a valid response
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=50, max=500, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Explain Python in one word.',
    model=MODEL,
    num_reasks=2
)
print('validated_output:', outcome.validated_output)
print('reask attempts:', len(guard.history[0].iterations) - 1)

## Example 04 — OnFailAction.FIX

In [ ]:
# Example 04: OnFailAction.FIX — validator auto-corrects the value
# RegexMatch with FIX replaces the non-matching string with fix_value if the validator supplies one
from guardrails.hub import RegexMatch
guard = Guard().use(RegexMatch(regex=r'^\d{3}-\d{4}$', on_fail=OnFailAction.FIX))
outcome = guard.validate('5551234')  # missing dash — validator may fix or flag
print('original: 5551234')
print('after FIX:', outcome.validated_output)
print('passed:', outcome.validation_passed)

## Example 05 — OnFailAction.FILTER

In [ ]:
# Example 05: OnFailAction.FILTER — the failing field is removed from output dict
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=5, max=50, on_fail=OnFailAction.FILTER))
outcome = guard.validate('Hi')  # too short
print('validated_output:', outcome.validated_output)  # None or filtered
print('validation_passed:', outcome.validation_passed)

## Example 06 — OnFailAction.REFRAIN

In [ ]:
# Example 06: OnFailAction.REFRAIN — entire response suppressed, returns None
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=20, max=500, on_fail=OnFailAction.REFRAIN))
outcome = guard.validate('Too short.')
print('validated_output:', outcome.validated_output)  # None
print('validation_passed:', outcome.validation_passed)

## Example 07 — OnFailAction.NOOP

In [ ]:
# Example 07: OnFailAction.NOOP — failure noted but value passes through unchanged
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=100, max=500, on_fail=OnFailAction.NOOP))
outcome = guard.validate('Short response.')
print('value passes through:', outcome.validated_output)  # original value returned
print('validation_passed:', outcome.validation_passed)    # False, but no exception

## Example 08 — OnFailAction.FIX_REASK

In [ ]:
# Example 08: OnFailAction.FIX_REASK — attempts fix first; reasks LLM only if fix insufficient
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=30, max=300, on_fail=OnFailAction.FIX_REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='What is 2+2? Answer in one word.',
    model=MODEL,
    num_reasks=1
)
print('validated_output:', outcome.validated_output)

## Example 09 — OnFailAction.CUSTOM

In [ ]:
# Example 09: OnFailAction.CUSTOM — supply a custom reask/fix callable
from guardrails.hub import ValidLength
from guardrails.actions.reask import FieldReAsk

def my_fix(value, fail_results):
    return value.strip() + ' [extended by custom action]'

guard = Guard().use(ValidLength(min=50, max=500, on_fail=my_fix))
outcome = guard.validate('Short text.')
print('custom-fixed output:', outcome.validated_output)

## Example 10 — ValidationOutcome Fields

In [ ]:
# Example 10: All key ValidationOutcome fields
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=5, max=200, on_fail=OnFailAction.NOOP))
outcome = guard.validate('Hello world, this is a test of validation outcome fields.')
print('validated_output :', outcome.validated_output)
print('validation_passed:', outcome.validation_passed)
print('raw_llm_output   :', outcome.raw_llm_output)
print('error            :', outcome.error)

## Example 11 — metadata= Parameter

In [ ]:
# Example 11: Pass a metadata dict that validators can read at runtime
# Useful for context-aware validators (provenance, saliency, etc.)
guard = Guard()
outcome = guard.validate(
    'The sky is blue.',
    metadata={'source_doc': 'Atmospheric Science 101', 'user_id': 'u42'}
)
print('passed with metadata:', outcome.validation_passed)

## Example 12 — validate() with a Plain String

In [ ]:
# Example 12: guard.validate() on a plain string (most common pattern)
from guardrails.hub import RegexMatch
guard = Guard().use(RegexMatch(regex=r'^[A-Z].*\.$', on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('starts lowercase')  # FAIL
except ValidationError:
    print('FAIL: does not start with capital or end with period')

outcome = guard.validate('This is correct.')
print('PASS:', outcome.validated_output)

## Example 13 — validate() with a Dict Input

In [ ]:
# Example 13: Validate a dict — each validator applies to the entire serialized dict
guard = Guard()
data = {'name': 'Alice', 'role': 'engineer', 'age': 30}
outcome = guard.validate(data)
print('dict validated:', outcome.validated_output)

## Example 14 — Guard History Inspection

In [ ]:
# Example 14: Inspect guard.history after LLM-integrated calls
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.NOOP))
guard(
    oai.chat.completions.create,
    prompt='Name a planet in one sentence.',
    model=MODEL
)
call = guard.history[0]
print('status :', call.status)
print('iterations:', len(call.iterations))
print('tokens_consumed:', call.tokens_consumed)

## Example 15 — Raw LLM Output Access

In [ ]:
# Example 15: Access raw (unvalidated) LLM response via history
guard = Guard()
guard(
    oai.chat.completions.create,
    prompt='What is the capital of France?',
    model=MODEL
)
raw = guard.history[0].iterations[0].raw_output
print('raw output:', raw)

## Example 16 — Validated Output Access

In [ ]:
# Example 16: Access the final validated output from history
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=5, max=300, on_fail=OnFailAction.NOOP))
guard(
    oai.chat.completions.create,
    prompt='Describe gravity in one sentence.',
    model=MODEL
)
validated = guard.history[0].validated_output
print('validated output:', validated)

## Example 17 — Error Detail Inspection

In [ ]:
# Example 17: Inspect error details in history after a NOOP fail
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=500, max=2000, on_fail=OnFailAction.NOOP))
guard.validate('This is too short for the validator.')
if guard.history:
    itr = guard.history[0].iterations[0]
    for ev in itr.validator_logs:
        print('validator:', ev.validator_name)
        print('result   :', ev.validation_result)

## Example 18 — num_reasks=3

In [ ]:
# Example 18: Set num_reasks to allow up to 3 retry attempts before giving up
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=200, max=2000, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Explain quantum entanglement briefly.',
    model=MODEL,
    num_reasks=3
)
print('final output length:', len(outcome.validated_output or ''))
print('total iterations   :', len(guard.history[0].iterations))

## Example 19 — Guard Re-use Across Multiple Calls

In [ ]:
# Example 19: Same guard instance used to validate multiple independent inputs
from guardrails.hub import RegexMatch
guard = Guard().use(RegexMatch(regex=r'^\d+$', on_fail=OnFailAction.EXCEPTION))
for text in ['42', '100', 'abc', '007']:
    try:
        guard.validate(text)
        print(f'  PASS: {text}')
    except ValidationError:
        print(f'  FAIL: {text}')

## Example 20 — Guard Serialization (to_dict / from_dict)

In [ ]:
# Example 20: Serialize a guard to dict and reconstruct it
from guardrails.hub import RegexMatch
original = Guard().use(RegexMatch(regex=r'^[A-Za-z]+$', on_fail=OnFailAction.EXCEPTION))
serialized = original.to_dict()
print('serialized keys:', list(serialized.keys()))
restored = Guard.from_dict(serialized)
outcome = restored.validate('Hello')
print('restored guard passes:', outcome.validation_passed)

## Example 21 — Guard Serialization to RAIL XML

In [ ]:
# Example 21: Serialize guard to legacy RAIL XML format
guard = Guard()
try:
    rail = guard.to_rail()
    print('RAIL snippet:', rail[:200])
except Exception as e:
    print('Note:', e)  # some guards need schema to produce RAIL

## Example 22 — Guard with Description Metadata

In [ ]:
# Example 22: Attach a human-readable description to a guard for documentation
guard = Guard(name='phone-validator', description='Validates North American phone numbers')
print('guard.name       :', guard.name)
print('guard.description:', guard.description)

## Example 23 — ValidationError Structure

In [ ]:
# Example 23: Inspect the ValidationError object in detail
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=50, max=500, on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Too short.')
except ValidationError as e:
    print('type :', type(e).__name__)
    print('str  :', str(e)[:200])

## Example 24 — Input Validation (pre-LLM)

In [ ]:
# Example 24: Validate user input before sending it to the LLM
from guardrails.hub import ValidLength
input_guard = Guard().use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))

user_input = 'How does photosynthesis work?'
try:
    input_guard.validate(user_input)
    resp = oai.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': user_input}]
    )
    print('LLM response:', resp.choices[0].message.content[:100])
except ValidationError as e:
    print('Input blocked:', e)

## Example 25 — Output Validation (post-LLM)

In [ ]:
# Example 25: Call LLM first, then validate the response
from guardrails.hub import ValidLength
resp = oai.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'List 3 programming languages.'}]
)
llm_output = resp.choices[0].message.content

output_guard = Guard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.EXCEPTION))
outcome = output_guard.validate(llm_output)
print('output passed validation:', outcome.validation_passed)
print('output:', outcome.validated_output[:100])

## Example 26 — Guard as LLM Callable Wrapper

In [ ]:
# Example 26: Use guard() as a drop-in wrapper around the LLM call
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=20, max=600, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Describe the water cycle.',
    model=MODEL,
    num_reasks=1
)
print('validated:', outcome.validated_output[:150])

## Example 27 — ValidationOutcome.validation_passed Status

In [ ]:
# Example 27: Check validation_passed boolean on pass and fail cases
from guardrails.hub import RegexMatch
guard = Guard().use(RegexMatch(regex=r'^\d{4}$', on_fail=OnFailAction.NOOP))
for text in ['1234', 'abcd', '99']:
    outcome = guard.validate(text)
    print(f'  input={text!r:6}  passed={outcome.validation_passed}')

## Example 28 — Streaming Guard

In [ ]:
# Example 28: Enable streaming and iterate over chunks
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=2000, on_fail=OnFailAction.NOOP))
fragment_generator = guard(
    oai.chat.completions.create,
    prompt='Explain machine learning in 2 sentences.',
    model=MODEL,
    stream=True
)
for chunk in fragment_generator:
    if chunk.validated_output:
        print(chunk.validated_output, end='', flush=True)
print()  # newline after stream

## Example 29 — REASK Loop with History Inspection

In [ ]:
# Example 29: Inspect each iteration of a REASK loop
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=100, max=1000, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Describe the solar system briefly.',
    model=MODEL,
    num_reasks=2
)
for i, itr in enumerate(guard.history[0].iterations):
    raw_len = len(itr.raw_output or '')
    print(f'  iteration {i}: raw_len={raw_len}')
print('final output length:', len(outcome.validated_output or ''))

## Example 30 — Custom Prompt Template

In [ ]:
# Example 30: Inject a custom prompt template into the guard
# The {output} placeholder is filled with the LLM's failing response on REASK
from guardrails.hub import ValidLength
custom_prompt = 'Your previous answer was too short. Please elaborate: {output}'
guard = Guard().use(ValidLength(min=100, max=2000, on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Name a fruit.',
    model=MODEL,
    num_reasks=1
)
print('outcome:', outcome.validated_output[:150] if outcome.validated_output else 'None')

## Example 31 — Multi-Field Dict Validation

In [ ]:
# Example 31: Validate a dict with multiple keys using Pydantic + Guard
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description='Product name')
    price: float = Field(description='Price in USD')
    in_stock: bool

guard = Guard.for_pydantic(output_class=Product)
outcome = guard.validate('{"name": "Widget", "price": 9.99, "in_stock": true}')
print('validated product:', outcome.validated_output)

## Example 32 — NOOP + Manual Post-Processing

In [ ]:
# Example 32: Use NOOP to collect failures, then handle manually in Python
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=50, max=500, on_fail=OnFailAction.NOOP))
texts = ['Hi.', 'A complete and well-formed sentence that is long enough to pass the minimum length check.', 'No.']
for text in texts:
    outcome = guard.validate(text)
    if not outcome.validation_passed:
        print(f'  [ACTION NEEDED] rejected ({len(text)} chars): {text[:40]}')
    else:
        print(f'  [OK] accepted: {text[:40]}')